In [0]:
SILVER = "workspace.silver.trips"
BRONZE = "workspace.bronze.trips"


def silver_select(year, month):
    return f"""
    SELECT
        VendorID                    AS vendor_id
      , tpep_pickup_datetime        AS pickup_at
      , tpep_dropoff_datetime       AS dropoff_at
      , CASE WHEN passenger_count = 0 THEN NULL ELSE CAST(passenger_count AS INT) END AS passenger_count
      , CAST(RatecodeID AS INT)     AS rate_code_id
      , trip_distance
      , store_and_fwd_flag
      , PULocationID                AS pickup_location_id
      , DOLocationID                AS dropoff_location_id
      , CAST(payment_type AS INT)   AS payment_type
      , fare_amount
      , extra
      , mta_tax
      , tip_amount
      , tolls_amount
      , improvement_surcharge
      , congestion_surcharge
      , airport_fee
      , total_amount
      , fare_amount < 0             AS is_negative_fare
      , trip_distance = 0           AS is_zero_distance
      , file_year
      , file_month
      , _source_file
      , _ingested_at
    FROM (
        SELECT
            *
          , year    AS file_year
          , month   AS file_month
        FROM
            {BRONZE}
        WHERE
            year  = {year}
        AND month = {month}
    ) src
    WHERE
        tpep_dropoff_datetime >= tpep_pickup_datetime
    AND year(tpep_pickup_datetime)  = file_year
    AND month(tpep_pickup_datetime) = file_month
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY
            VendorID
          , tpep_pickup_datetime
          , tpep_dropoff_datetime
          , PULocationID
          , DOLocationID
          , total_amount
        ORDER BY
            _ingested_at
    ) = 1
    """

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

# first run ever: create an empty table with the right columns
if not spark.catalog.tableExists(SILVER):
    spark.sql(f"CREATE TABLE {SILVER} AS {silver_select(0, 0)}")

new_months = spark.sql(f"""
    SELECT
        b.year
      , b.month
    FROM
        (SELECT year, month, MAX(_ingested_at) AS loaded_at FROM {BRONZE} GROUP BY year, month) b
    LEFT JOIN
        (SELECT file_year, file_month, MAX(_ingested_at) AS loaded_at FROM {SILVER} GROUP BY file_year, file_month) s
        ON  b.year  = s.file_year
        AND b.month = s.file_month
    WHERE
        s.loaded_at IS NULL
    OR  b.loaded_at > s.loaded_at
    ORDER BY
        b.year, b.month
""").collect()

for row in new_months:
    spark.sql(f"""
        INSERT INTO {SILVER}
        REPLACE WHERE file_year = {row.year} AND file_month = {row.month}
        {silver_select(row.year, row.month)}
    """)
    print(f"loaded {row.year}-{row.month:02d}")

if not new_months:
    print("silver is up to date")